[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C45_Privacy_Trustworthy_Course/00_setup/00_environment_check.ipynb)

# 00 · 环境自检与方法论热身

本课全程 **纯 numpy、CPU 可跑**，用 numpy **从零实现隐私机制**（加噪、裁剪、聚合、遗忘），再与 **可信参照** 对拍。

这个 notebook 做四件事：① 确认环境；② 用一个最小例子体会「**模型会泄露训练个体**」这个根本威胁；③ 立下全课纪律——**对拍（differential testing）**；④ 先把后面每个模块都要用的 **隐私-效用权衡** 直觉建起来。

## 1 · 环境自检

只需要 `numpy`。`matplotlib` 可选（仅用于画隐私-效用曲线）。

In [ ]:
import sys, platform
print('Python', sys.version.split()[0], '|', platform.system())
import numpy as np
print('numpy', np.__version__)
try:
    import matplotlib; print('matplotlib', matplotlib.__version__, '(可选)')
except Exception:
    print('matplotlib 未安装（可选，不影响课程）')
print('环境就绪 ✅')

## 2 · 根本威胁：模型会「记住」训练个体

隐私的根源是：模型对**见过的**样本，往往比对**没见过的**样本更自信。这正是成员推断攻击（membership inference）的抓手。

我们用一个最小例子演示：在小数据上把模型训得「太好」（过拟合），见过的样本损失会明显低于没见过的——这个差距就是隐私泄露的指纹。

In [ ]:
rng = np.random.default_rng(0)

# 一个会过拟合的玩具：用高维线性模型记住少量样本
n_train, d = 20, 40        # 样本少、维度高 -> 容易记住
X_train = rng.standard_normal((n_train, d))
w_true = rng.standard_normal(d)
y_train = X_train @ w_true + 0.1 * rng.standard_normal(n_train)

# 最小二乘（无正则）-> 在 n<d 时能近乎完美拟合训练集（记住它们）
w_hat, *_ = np.linalg.lstsq(X_train, y_train, rcond=None)

# 训练样本 vs 全新样本 的预测误差
X_new = rng.standard_normal((n_train, d))
y_new = X_new @ w_true + 0.1 * rng.standard_normal(n_train)
err_train = np.mean((X_train @ w_hat - y_train) ** 2)
err_new   = np.mean((X_new   @ w_hat - y_new)   ** 2)
print(f'训练样本均方误差 = {err_train:.4f}  (模型见过它们)')
print(f'全新样本均方误差 = {err_new:.4f}  (模型没见过)')
print(f'差距(泄露指纹)   = {err_new/max(err_train,1e-12):.0f}x')
assert err_train < err_new, '过拟合时训练样本误差应明显更小 —— 这就是成员推断的抓手'
print('✅ 见过/没见过 的差距 = 隐私泄露的根源。DP 的使命就是压制这个差距。')

## 3 · 立纪律：对拍（differential testing）

本课每个隐私机制都要和一个**可信参照**比对。最常用的参照之一是**理论期望/方差**：一个无偏的加噪机制，多次平均应回到真值。

先把这个工作流跑通：写一个最朴素的「加高斯噪声」机制，验证它**无偏**（期望=真值）且**噪声尺度正确**（方差≈设定值）。

In [ ]:
def noisy_mean(values, sigma, rng):
    '''最朴素的加噪：真实均值 + 高斯噪声。后面 DP 机制都是它的严格版。'''
    true_mean = np.mean(values)
    return true_mean + rng.normal(0.0, sigma)

data = rng.standard_normal(100)
true_mean = data.mean()
sigma = 0.5
# 跑很多次，看带噪结果的均值是否回到真值、方差是否≈sigma^2
draws = np.array([noisy_mean(data, sigma, rng) for _ in range(20000)])
print(f'真实均值         = {true_mean:.4f}')
print(f'带噪结果的均值   = {draws.mean():.4f}  (应≈真值 -> 无偏)')
print(f'带噪结果的方差   = {draws.var():.4f}  (应≈sigma^2={sigma**2:.4f})')
assert abs(draws.mean() - true_mean) < 0.02, '加噪机制应无偏'
assert abs(draws.var() - sigma**2) < 0.02, '噪声方差应≈设定值'
print('✅ 对拍理论通过：无偏 + 噪声尺度正确。这是全课验证 DP 机制的基本套路。')

## 4 · 隐私-效用权衡：没有免费的隐私

DP 的中心事实：**噪声越大越私密，但结果越不准**。我们提前把这条权衡曲线画出来（用数字，不依赖 matplotlib）。

对同一个查询（数据均值），用不同噪声尺度 `sigma` 估计，看「误差」如何随「隐私强度」上升——这是后面每个模块都要面对的取舍。

In [ ]:
def utility_error(values, sigma, rng, trials=2000):
    '''给定噪声尺度，估计带噪均值相对真值的平均绝对误差（效用损失的代理）。'''
    true_mean = np.mean(values)
    errs = [abs(noisy_mean(values, sigma, rng) - true_mean) for _ in range(trials)]
    return np.mean(errs)

data = rng.standard_normal(1000)
print(f"{'噪声sigma':>10s} {'平均|误差|':>12s}  {'解读':<18s}")
prev = -1.0
for sigma in [0.05, 0.1, 0.2, 0.5, 1.0]:
    e = utility_error(data, sigma, rng)
    tag = '更私密/更不准' if sigma >= 0.5 else '更准/更不私密'
    print(f'{sigma:>10.2f} {e:>12.4f}  {tag:<18s}')
    assert e > prev, '噪声越大，误差应单调上升（隐私换效用）'
    prev = e
print('\n✅ 误差随噪声单调上升 —— 这就是隐私-效用权衡。DP 把它量化成 ε 与噪声的关系。')

## 5 · 一个会贯穿全课的对拍工具

把「对拍」封装成统一裁判，后面每个模块都用它判定「我的机制 == 参照」。

In [ ]:
def check_close(name, got, ref, atol=1e-8):
    '''对拍：机制结果 vs 可信参照。打印并 assert。'''
    got_a, ref_a = np.asarray(got, dtype=float), np.asarray(ref, dtype=float)
    ok = np.allclose(got_a, ref_a, atol=atol)
    max_err = float(np.max(np.abs(got_a - ref_a))) if got_a.size else 0.0
    print(f'[{name:<28}] close={ok}  max|err|={max_err:.2e}')
    assert ok, f'{name} 与参照不一致！'
    return ok

def check_within(name, value, lo, hi):
    '''对拍：某个量（如经验 ε、效用代价）落在预期区间内。'''
    ok = lo <= value <= hi
    print(f'[{name:<28}] value={value:.4f}  in [{lo}, {hi}] -> {ok}')
    assert ok, f'{name}={value} 不在预期区间 [{lo},{hi}]'
    return ok

# 演示：一个无偏加噪机制，大量平均后应落在真值附近
data = rng.standard_normal(500)
avg_of_noisy = np.mean([noisy_mean(data, 0.3, rng) for _ in range(5000)])
check_within('无偏机制均值≈真值', avg_of_noisy, data.mean()-0.02, data.mean()+0.02)
print('\n这就是全课的工作流：写机制 -> 对拍参照(理论/重训/集中式) -> assert 兜底。')

✅ 检查全部通过即环境就绪、方法论到位。

**本课的契约**：你在 numpy 里写出的每个 DP / 联邦 / 遗忘机制，都会用对拍验证——对拍理论期望、对拍非私有训练、对拍集中式、对拍重训；机制正确则数值符合，数值符合则逻辑可迁移到 Opacus / Flower。

**接下来六个模块**：01 差分隐私 → 02 DP-SGD → 03 联邦学习 → 04 机器遗忘 → 05 可信部署。前四个给工具，第五个拼成系统。

下一站：**模块 01 · 差分隐私** —— 给隐私一个数学定义和可累加的预算。